<a href="https://colab.research.google.com/github/mkobycheva/recommendation-system/blob/agent_als/notebooks/01_baselines_with_agent_ratings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ALS Baseline: Books + Movies Intersection

Train one implicit ALS model on merged Books and Movies interactions from the prepared Google Drive splits. Recommendations are filtered back to the target domain before MAP@10 evaluation.

In [1]:
!test -d /content/recommendation-system/.git \
  || git clone -b als https://github.com/mkobycheva/recommendation-system.git /content/recommendation-system

%cd /content/recommendation-system
!git fetch origin main
!git checkout main
!git reset --hard origin/main

# Create and switch to the new agent_als branch
!git checkout -b agent_als

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

Cloning into '/content/recommendation-system'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 257 (delta 17), reused 20 (delta 6), pack-reused 212 (from 1)
Receiving objects: 100% (257/257), 665.04 KiB | 25.58 MiB/s, done.
Resolving deltas: 100% (133/133), done.
/content/recommendation-system
From https://github.com/mkobycheva/recommendation-system
 * branch            main       -> FETCH_HEAD
Branch 'main' set up to track remote branch 'main' from 'origin'.
Switched to a new branch 'main'
HEAD is now at 6fef0c2 Merge pull request #4 from mkobycheva/als
Switched to a new branch 'agent_als'
Mounted at /content/drive


In [2]:
!pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 98.8 MB/s eta 0:00:00


In [3]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import implicit
import numpy as np
import pandas as pd

from src.data.build_matrix import add_domain_item_ids, build_implicit_als_matrix
from src.evaluation.metrics import map_at_k, ndcg_at_k, recall_at_k
from src.evaluation.cross_domain_eval import evaluate_multi_domain

/usr/local/lib/python3.12/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


## Load Prepared Splits

The split CSVs are expected to contain the Books/Movies intersecting users and time-based train/validation/test rows.

In [4]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [5]:
books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()

,user_id,item_id,domain
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0061713244,books
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0871139634,books
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0385521685,books
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0848732634,books
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0151014981,books


## Shared Indexes and Train Matrix

In [6]:
interaction_matrix = build_implicit_als_matrix(train_df)

user_item_train = interaction_matrix.user_item_train
user2idx = interaction_matrix.user2idx
item2idx = interaction_matrix.item2idx
idx2item = interaction_matrix.idx2item
item_domain = interaction_matrix.item_domain
train_seen_idx_by_user = interaction_matrix.train_seen_idx_by_user
domain_item_indices = interaction_matrix.domain_item_indices

n_users, n_items = user_item_train.shape

print(f'users={n_users:,}, items={n_items:,}, train_interactions={user_item_train.nnz:,}')
print(train_df['domain'].value_counts())

users=127,188, items=587,064, train_interactions=3,750,813
domain
movies    1887583
books     1863230
Name: count, dtype: int64


In [7]:
assert user_item_train.shape == (n_users, n_items)
assert len(domain_item_indices['books']) > 0
assert len(domain_item_indices['movies']) > 0

## Train Collective ALS

In [8]:
FACTORS = 128          # Increased from 64
REGULARIZATION = 0.05  # Increased from 0.01
ITERATIONS = 20        # Kept at 20
ALPHA = 15             # Lowered from 40
K = 10

als_model = implicit.als.AlternatingLeastSquares(
    factors=FACTORS,
    regularization=REGULARIZATION,
    iterations=ITERATIONS,
    random_state=42,
)

item_user_train = (user_item_train * ALPHA).T.tocsr()
als_model.fit(item_user_train)

# The model is fit on item-user input, so implicit stores item vectors in
# user_factors and user vectors in item_factors.
als_item_factors = als_model.user_factors
als_user_factors = als_model.item_factors

assert als_item_factors.shape[0] == n_items
assert als_user_factors.shape[0] == n_users

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

## Domain-Filtered Recommendations

In [9]:
_candidate_to_position = {
    domain: {item_idx: pos for pos, item_idx in enumerate(indices)}
    for domain, indices in domain_item_indices.items()
}

def relevant_items_by_user(split_df, target_domain):
    domain_rows = split_df[split_df['domain'] == target_domain]
    return domain_rows.groupby('user_id')['item_id'].agg(set).to_dict()


def recommend_for_users(user_ids, target_domain, k=10, batch_size=512, max_score_elements=20_000_000):
    candidates = domain_item_indices[target_domain]
    candidate_factors = als_item_factors[candidates]
    candidate_to_position = _candidate_to_position[target_domain]  # O(1) instant reuse

    recommendations = {}
    user_ids = list(user_ids)
    effective_batch_size = max(1, min(batch_size, max_score_elements // max(len(candidates), 1)))

    for batch_start in range(0, len(user_ids), effective_batch_size):
        batch = user_ids[batch_start:batch_start + effective_batch_size]
        valid_user_ids = []
        user_indices = []

        for user_id in batch:
            user_idx = user2idx.get(user_id)
            if user_idx is None:
                recommendations[user_id] = []
                continue
            valid_user_ids.append(user_id)
            user_indices.append(user_idx)

        if not user_indices:
            continue

        user_factors = als_user_factors[np.array(user_indices, dtype=np.int64)]

        # Compute scores for the entire batch instantly
        all_scores = user_factors @ candidate_factors.T

        # VECTORIZED MASKING: Extract history blocks for the whole batch at once
        for idx, user_idx in enumerate(user_indices):
            seen_items = train_seen_idx_by_user.get(user_idx, set())
            # Fast list comprehension via local O(1) dictionary lookups
            known_pos = [candidate_to_position[i] for i in seen_items if i in candidate_to_position]
            if known_pos:
                all_scores[idx, known_pos] = -np.inf

        # Run argpartition on the fully masked matrix in parallel
        for row_idx, user_id in enumerate(valid_user_ids):
            scores = all_scores[row_idx]
            finite_count = np.isfinite(scores).sum()
            if finite_count == 0:
                recommendations[user_id] = []
                continue

            top_n = min(k, int(finite_count))
            top_positions = np.argpartition(-scores, top_n - 1)[:top_n]
            top_positions = top_positions[np.argsort(-scores[top_positions])]
            recommendations[user_id] = [idx2item[int(candidates[pos])] for pos in top_positions]

        if (batch_start // effective_batch_size) % 20 == 0:
            print(f'  Processed {batch_start:,}/{len(user_ids):,} users for {target_domain}...')

    return recommendations

## Evaluate and Save Metrics

In [10]:
K = 10

# 1. Clear interface router mapping to notebook inference algorithm
def run_als_inference(user_ids, target_domain, k=10):
    return recommend_for_users(user_ids, target_domain, k=k)

# 2. Unified, safe execution sweep across splits simultaneously
valid_results, test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=run_als_inference,
    k=K
)

# 3. Correctly label model payload before logging to CSV
os.makedirs('results', exist_ok=True)
result_row = {
    'model': 'implicit_als_books_movies_joint_weighted',  # Corrected name label
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': n_users,
    'n_items': n_items,
    'n_train_interactions': user_item_train.nnz,
    'books_valid_ndcg@10': valid_results['books']['ndcg@10'],
    'movies_valid_ndcg@10': valid_results['movies']['ndcg@10'],
    'valid_macro_ndcg@10': round((valid_results['books']['ndcg@10'] + valid_results['movies']['ndcg@10']) / 2, 4),
    'books_valid_recall@10': valid_results['books']['recall@10'],
    'movies_valid_recall@10': valid_results['movies']['recall@10'],
    'valid_macro_recall@10': round((valid_results['books']['recall@10'] + valid_results['movies']['recall@10']) / 2, 4),
    'books_valid_map@10': valid_results['books']['map@10'],
    'movies_valid_map@10': valid_results['movies']['map@10'],
    'valid_macro_map@10': round((valid_results['books']['map@10'] + valid_results['movies']['map@10']) / 2, 4),
    'books_test_ndcg@10': test_results['books']['ndcg@10'],
    'movies_test_ndcg@10': test_results['movies']['ndcg@10'],
    'test_macro_ndcg@10': round((test_results['books']['ndcg@10'] + test_results['movies']['ndcg@10']) / 2, 4),
    'books_test_recall@10': test_results['books']['recall@10'],
    'movies_test_recall@10': test_results['movies']['recall@10'],
    'test_macro_recall@10': round((test_results['books']['recall@10'] + test_results['movies']['recall@10']) / 2, 4),
    'books_test_map@10': test_results['books']['map@10'],
    'movies_test_map@10': test_results['movies']['map@10'],
    'test_macro_map@10': round((test_results['books']['map@10'] + test_results['movies']['map@10']) / 2, 4),
}

results_df = pd.DataFrame([result_row])
results_df.to_csv('results/als_baseline_metrics.csv', index=False)
results_df

[Evaluation] Scoring BOOKS: 127,188 unique users across splits
  Processed 0/127,188 users for books...
  Processed 980/127,188 users for books...
  Processed 1,960/127,188 users for books...
  Processed 2,940/127,188 users for books...
  Processed 3,920/127,188 users for books...
  Processed 4,900/127,188 users for books...
  Processed 5,880/127,188 users for books...
  Processed 6,860/127,188 users for books...
  Processed 7,840/127,188 users for books...
  Processed 8,820/127,188 users for books...
  Processed 9,800/127,188 users for books...
  Processed 10,780/127,188 users for books...
  Processed 11,760/127,188 users for books...
  Processed 12,740/127,188 users for books...
  Processed 13,720/127,188 users for books...
  Processed 14,700/127,188 users for books...
  Processed 15,680/127,188 users for books...
  Processed 16,660/127,188 users for books...
  Processed 17,640/127,188 users for books...
  Processed 18,620/127,188 users for books...
  Processed 19,600/127,188 users f

,model,factors,regularization,iterations,alpha,k,n_users,n_items,n_train_interactions,books_valid_ndcg@10,...,valid_macro_map@10,books_test_ndcg@10,movies_test_ndcg@10,test_macro_ndcg@10,books_test_recall@10,movies_test_recall@10,test_macro_recall@10,books_test_map@10,movies_test_map@10,test_macro_map@10
0,implicit_als_books_movies_joint_weighted,128,0.05,20,15,10,127188,587064,3750813,0.0109,...,0.0124,0.0073,0.0127,0.01,0.0138,0.0249,0.0193,0.0054,0.0091,0.0073


## Ablation: Single-Domain ALS

Train separate Books-only and Movies-only ALS models with the same hyperparameters as the collective model, then compare in-domain ranking performance against the collective ALS baseline.

In [11]:
def train_single_domain_als(domain_train_df):
    domain_matrix = build_implicit_als_matrix(domain_train_df)
    domain_user_item_train = domain_matrix.user_item_train

    domain_model = implicit.als.AlternatingLeastSquares(
        factors=FACTORS,
        regularization=REGULARIZATION,
        iterations=ITERATIONS,
        random_state=42,
    )

    domain_item_user_train = (domain_user_item_train * ALPHA).T.tocsr()
    domain_model.fit(domain_item_user_train)

    return {
        'model': domain_model,
        'matrix': domain_matrix,
        'user_item_train': domain_user_item_train,
        'item_factors': domain_model.user_factors,
        'user_factors': domain_model.item_factors,
        'candidate_to_position': {
            domain: {item_idx: pos for pos, item_idx in enumerate(indices)}
            for domain, indices in domain_matrix.domain_item_indices.items()
        },
    }


def recommend_single_domain_als(
    user_ids,
    target_domain,
    *,
    trained_domain_model,
    k=10,
    batch_size=512,
    max_score_elements=20_000_000,
):
    domain_matrix = trained_domain_model['matrix']
    candidates = domain_matrix.domain_item_indices[target_domain]
    candidate_factors = trained_domain_model['item_factors'][candidates]
    candidate_to_position = trained_domain_model['candidate_to_position'][target_domain]

    recommendations = {}
    user_ids = list(user_ids)
    effective_batch_size = max(1, min(batch_size, max_score_elements // max(len(candidates), 1)))

    for batch_start in range(0, len(user_ids), effective_batch_size):
        batch = user_ids[batch_start:batch_start + effective_batch_size]
        valid_user_ids = []
        user_indices = []

        for user_id in batch:
            user_idx = domain_matrix.user2idx.get(user_id)
            if user_idx is None:
                recommendations[user_id] = []
                continue
            valid_user_ids.append(user_id)
            user_indices.append(user_idx)

        if not user_indices:
            continue

        user_factors = trained_domain_model['user_factors'][np.array(user_indices, dtype=np.int64)]
        all_scores = user_factors @ candidate_factors.T

        for idx, user_idx in enumerate(user_indices):
            seen_items = domain_matrix.train_seen_idx_by_user.get(user_idx, set())
            known_pos = [candidate_to_position[i] for i in seen_items if i in candidate_to_position]
            if known_pos:
                all_scores[idx, known_pos] = -np.inf

        for row_idx, user_id in enumerate(valid_user_ids):
            scores = all_scores[row_idx]
            finite_count = np.isfinite(scores).sum()
            if finite_count == 0:
                recommendations[user_id] = []
                continue

            top_n = min(k, int(finite_count))
            top_positions = np.argpartition(-scores, top_n - 1)[:top_n]
            top_positions = top_positions[np.argsort(-scores[top_positions])]
            recommendations[user_id] = [domain_matrix.idx2item[int(candidates[pos])] for pos in top_positions]

        if (batch_start // effective_batch_size) % 20 == 0:
            print(f'  Processed {batch_start:,}/{len(user_ids):,} users for {target_domain}...')

    return recommendations


In [12]:
books_only_model = train_single_domain_als(books_train)

def run_books_only_als_inference(user_ids, target_domain, k=10):
    return recommend_single_domain_als(
        user_ids,
        target_domain,
        trained_domain_model=books_only_model,
        k=k,
    )

books_only_valid_results, books_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'books'],
        'test': test_df[test_df['domain'] == 'books'],
    },
    recommend_func=run_books_only_als_inference,
    k=K,
)

books_only_result_row = {
    'model': 'implicit_als_books_only_weighted',
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': books_only_model['user_item_train'].shape[0],
    'n_items': books_only_model['user_item_train'].shape[1],
    'n_train_interactions': books_only_model['user_item_train'].nnz,
    'books_valid_ndcg@10': books_only_valid_results['books']['ndcg@10'],
    'movies_valid_ndcg@10': np.nan,
    'valid_macro_ndcg@10': books_only_valid_results['books']['ndcg@10'],
    'books_valid_recall@10': books_only_valid_results['books']['recall@10'],
    'movies_valid_recall@10': np.nan,
    'valid_macro_recall@10': books_only_valid_results['books']['recall@10'],
    'books_valid_map@10': books_only_valid_results['books']['map@10'],
    'movies_valid_map@10': np.nan,
    'valid_macro_map@10': books_only_valid_results['books']['map@10'],
    'books_test_ndcg@10': books_only_test_results['books']['ndcg@10'],
    'movies_test_ndcg@10': np.nan,
    'test_macro_ndcg@10': books_only_test_results['books']['ndcg@10'],
    'books_test_recall@10': books_only_test_results['books']['recall@10'],
    'movies_test_recall@10': np.nan,
    'test_macro_recall@10': books_only_test_results['books']['recall@10'],
    'books_test_map@10': books_only_test_results['books']['map@10'],
    'movies_test_map@10': np.nan,
    'test_macro_map@10': books_only_test_results['books']['map@10'],
}

books_only_result_row


  0%|          | 0/20 [00:00<?, ?it/s]

[Evaluation] Scoring BOOKS: 127,188 unique users across splits
  Processed 0/127,188 users for books...
  Processed 980/127,188 users for books...
  Processed 1,960/127,188 users for books...
  Processed 2,940/127,188 users for books...
  Processed 3,920/127,188 users for books...
  Processed 4,900/127,188 users for books...
  Processed 5,880/127,188 users for books...
  Processed 6,860/127,188 users for books...
  Processed 7,840/127,188 users for books...
  Processed 8,820/127,188 users for books...
  Processed 9,800/127,188 users for books...
  Processed 10,780/127,188 users for books...
  Processed 11,760/127,188 users for books...
  Processed 12,740/127,188 users for books...
  Processed 13,720/127,188 users for books...
  Processed 14,700/127,188 users for books...
  Processed 15,680/127,188 users for books...
  Processed 16,660/127,188 users for books...
  Processed 17,640/127,188 users for books...
  Processed 18,620/127,188 users for books...
  Processed 19,600/127,188 users f

{'model': 'implicit_als_books_only_weighted',
 'factors': 128,
 'regularization': 0.05,
 'iterations': 20,
 'alpha': 15,
 'k': 10,
 'n_users': 127188,
 'n_items': 404630,
 'n_train_interactions': 1863230,
 'books_valid_ndcg@10': 0.0162,
 'movies_valid_ndcg@10': nan,
 'valid_macro_ndcg@10': 0.0162,
 'books_valid_recall@10': 0.0299,
 'movies_valid_recall@10': nan,
 'valid_macro_recall@10': 0.0299,
 'books_valid_map@10': 0.0121,
 'movies_valid_map@10': nan,
 'valid_macro_map@10': 0.0121,
 'books_test_ndcg@10': 0.0101,
 'movies_test_ndcg@10': nan,
 'test_macro_ndcg@10': 0.0101,
 'books_test_recall@10': 0.0186,
 'movies_test_recall@10': nan,
 'test_macro_recall@10': 0.0186,
 'books_test_map@10': 0.0075,
 'movies_test_map@10': nan,
 'test_macro_map@10': 0.0075}

In [13]:
movies_only_model = train_single_domain_als(movies_train)

def run_movies_only_als_inference(user_ids, target_domain, k=10):
    return recommend_single_domain_als(
        user_ids,
        target_domain,
        trained_domain_model=movies_only_model,
        k=k,
    )

movies_only_valid_results, movies_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'movies'],
        'test': test_df[test_df['domain'] == 'movies'],
    },
    recommend_func=run_movies_only_als_inference,
    k=K,
)

movies_only_result_row = {
    'model': 'implicit_als_movies_only_weighted',
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': movies_only_model['user_item_train'].shape[0],
    'n_items': movies_only_model['user_item_train'].shape[1],
    'n_train_interactions': movies_only_model['user_item_train'].nnz,
    'books_valid_ndcg@10': np.nan,
    'movies_valid_ndcg@10': movies_only_valid_results['movies']['ndcg@10'],
    'valid_macro_ndcg@10': movies_only_valid_results['movies']['ndcg@10'],
    'books_valid_recall@10': np.nan,
    'movies_valid_recall@10': movies_only_valid_results['movies']['recall@10'],
    'valid_macro_recall@10': movies_only_valid_results['movies']['recall@10'],
    'books_valid_map@10': np.nan,
    'movies_valid_map@10': movies_only_valid_results['movies']['map@10'],
    'valid_macro_map@10': movies_only_valid_results['movies']['map@10'],
    'books_test_ndcg@10': np.nan,
    'movies_test_ndcg@10': movies_only_test_results['movies']['ndcg@10'],
    'test_macro_ndcg@10': movies_only_test_results['movies']['ndcg@10'],
    'books_test_recall@10': np.nan,
    'movies_test_recall@10': movies_only_test_results['movies']['recall@10'],
    'test_macro_recall@10': movies_only_test_results['movies']['recall@10'],
    'books_test_map@10': np.nan,
    'movies_test_map@10': movies_only_test_results['movies']['map@10'],
    'test_macro_map@10': movies_only_test_results['movies']['map@10'],
}

movies_only_result_row


  0%|          | 0/20 [00:00<?, ?it/s]

[Evaluation] Scoring MOVIES: 127,188 unique users across splits
  Processed 0/127,188 users for movies...
  Processed 2,180/127,188 users for movies...
  Processed 4,360/127,188 users for movies...
  Processed 6,540/127,188 users for movies...
  Processed 8,720/127,188 users for movies...
  Processed 10,900/127,188 users for movies...
  Processed 13,080/127,188 users for movies...
  Processed 15,260/127,188 users for movies...
  Processed 17,440/127,188 users for movies...
  Processed 19,620/127,188 users for movies...
  Processed 21,800/127,188 users for movies...
  Processed 23,980/127,188 users for movies...
  Processed 26,160/127,188 users for movies...
  Processed 28,340/127,188 users for movies...
  Processed 30,520/127,188 users for movies...
  Processed 32,700/127,188 users for movies...
  Processed 34,880/127,188 users for movies...
  Processed 37,060/127,188 users for movies...
  Processed 39,240/127,188 users for movies...
  Processed 41,420/127,188 users for movies...
  Pro

{'model': 'implicit_als_movies_only_weighted',
 'factors': 128,
 'regularization': 0.05,
 'iterations': 20,
 'alpha': 15,
 'k': 10,
 'n_users': 127188,
 'n_items': 182434,
 'n_train_interactions': 1887583,
 'books_valid_ndcg@10': nan,
 'movies_valid_ndcg@10': 0.0272,
 'valid_macro_ndcg@10': 0.0272,
 'books_valid_recall@10': nan,
 'movies_valid_recall@10': 0.049,
 'valid_macro_recall@10': 0.049,
 'books_valid_map@10': nan,
 'movies_valid_map@10': 0.0206,
 'valid_macro_map@10': 0.0206,
 'books_test_ndcg@10': nan,
 'movies_test_ndcg@10': 0.014,
 'test_macro_ndcg@10': 0.014,
 'books_test_recall@10': nan,
 'movies_test_recall@10': 0.0268,
 'test_macro_recall@10': 0.0268,
 'books_test_map@10': nan,
 'movies_test_map@10': 0.0101,
 'test_macro_map@10': 0.0101}

In [14]:
collective_result_row = {
    'model': 'implicit_als_books_movies_joint_weighted',
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': n_users,
    'n_items': n_items,
    'n_train_interactions': user_item_train.nnz,
    'books_valid_ndcg@10': valid_results['books']['ndcg@10'],
    'movies_valid_ndcg@10': valid_results['movies']['ndcg@10'],
    'valid_macro_ndcg@10': round((valid_results['books']['ndcg@10'] + valid_results['movies']['ndcg@10']) / 2, 4),
    'books_valid_recall@10': valid_results['books']['recall@10'],
    'movies_valid_recall@10': valid_results['movies']['recall@10'],
    'valid_macro_recall@10': round((valid_results['books']['recall@10'] + valid_results['movies']['recall@10']) / 2, 4),
    'books_valid_map@10': valid_results['books']['map@10'],
    'movies_valid_map@10': valid_results['movies']['map@10'],
    'valid_macro_map@10': round((valid_results['books']['map@10'] + valid_results['movies']['map@10']) / 2, 4),
    'books_test_ndcg@10': test_results['books']['ndcg@10'],
    'movies_test_ndcg@10': test_results['movies']['ndcg@10'],
    'test_macro_ndcg@10': round((test_results['books']['ndcg@10'] + test_results['movies']['ndcg@10']) / 2, 4),
    'books_test_recall@10': test_results['books']['recall@10'],
    'movies_test_recall@10': test_results['movies']['recall@10'],
    'test_macro_recall@10': round((test_results['books']['recall@10'] + test_results['movies']['recall@10']) / 2, 4),
    'books_test_map@10': test_results['books']['map@10'],
    'movies_test_map@10': test_results['movies']['map@10'],
    'test_macro_map@10': round((test_results['books']['map@10'] + test_results['movies']['map@10']) / 2, 4),
}

als_ablation_results = pd.DataFrame([
    books_only_result_row,
    movies_only_result_row,
    collective_result_row,
])

als_ablation_comparison = als_ablation_results.set_index('model')[[
    'books_test_ndcg@10',
    'books_test_recall@10',
    'books_test_map@10',
    'movies_test_ndcg@10',
    'movies_test_recall@10',
    'movies_test_map@10',
]].rename(index={
    'implicit_als_books_only_weighted': 'Books-only',
    'implicit_als_movies_only_weighted': 'Movies-only',
    'implicit_als_books_movies_joint_weighted': 'Collective',
}).rename(columns={
    'books_test_ndcg@10': 'Books NDCG@10',
    'books_test_recall@10': 'Books Recall@10',
    'books_test_map@10': 'Books MAP@10',
    'movies_test_ndcg@10': 'Movies NDCG@10',
    'movies_test_recall@10': 'Movies Recall@10',
    'movies_test_map@10': 'Movies MAP@10',
})

os.makedirs('results', exist_ok=True)
als_ablation_comparison.to_csv('results/als_ablation_comparison.csv')
als_ablation_comparison


,Books NDCG@10,Books Recall@10,Books MAP@10,Movies NDCG@10,Movies Recall@10,Movies MAP@10
model,,,,,,
Books-only,0.0101,0.0186,0.0075,NaN,NaN,NaN
Movies-only,NaN,NaN,NaN,0.0140,0.0268,0.0101
Collective,0.0073,0.0138,0.0054,0.0127,0.0249,0.0091


# Here starts work with agents

Sampling users and building the per-user record (history + model recommendations)

In [45]:
import random
import pandas as pd

SAMPLE_SEED = 676767
N_USERS_PER_DOMAIN = 100
CANDIDATE_POOL_MULTIPLIER = 5  # draw extra candidates upfront since ~40% of movies get dropped

DATA_DIR = '/content/drive/MyDrive/recsys-data'
books_meta = pd.read_csv(f'{DATA_DIR}/books_meta.csv')
movies_meta = pd.read_csv(f'{DATA_DIR}/movies_meta.csv')

books_title = books_meta.set_index('parent_asin')['title'].to_dict()
movies_title = movies_meta.set_index('parent_asin')['title'].to_dict()

def title_for(item_id):
    domain, asin = item_id.split('::', 1)
    lookup = books_title if domain == 'books' else movies_title
    return lookup.get(asin, '(title not found)')

def has_valid_title(item_id):
    domain, asin = item_id.split('::', 1)
    lookup = books_title if domain == 'books' else movies_title
    title = lookup.get(asin)
    return isinstance(title, str)  # excludes NaN (float) and missing keys (None)

def sample_candidate_users(target_domain, n, seed):
    domain_test_users = test_df.loc[test_df['domain'] == target_domain, 'user_id'].unique().tolist()
    rng = random.Random(seed)
    rng.shuffle(domain_test_users)
    return domain_test_users[:n]

def user_history_items(user_id, target_domain):
    user_idx = user2idx.get(user_id)
    seen_idx = train_seen_idx_by_user.get(user_idx, set())
    return [idx2item[i] for i in seen_idx if item_domain[i] == target_domain][:8]

def build_clean_eval_records(target_domain, n_target=N_USERS_PER_DOMAIN, seed=SAMPLE_SEED):
    candidates = sample_candidate_users(target_domain, n_target * CANDIDATE_POOL_MULTIPLIER, seed)

    # Cheap local filter first: keep only users whose history is non-empty
    # and every history item has a real title. No model calls yet.
    pre_filtered = []
    for user_id in candidates:
        history_items = user_history_items(user_id, target_domain)
        if history_items and all(has_valid_title(i) for i in history_items):
            pre_filtered.append(user_id)
        if len(pre_filtered) >= n_target * 2:  # small buffer for the rec-side filter below
            break

    print(f"{target_domain}: {len(pre_filtered)} candidates passed history filter (from {len(candidates)} drawn)")

    # Now get recommendations for the filtered pool, then drop anyone whose
    # rec list still contains a title-less item, until we hit n_target.
    recs_by_user = recommend_for_users(pre_filtered, target_domain, k=10)
    ground_truth = test_df[(test_df['domain'] == target_domain) & (test_df['user_id'].isin(pre_filtered))] \
        .groupby('user_id')['item_id'].agg(list).to_dict()

    records = []
    for user_id in pre_filtered:
        recs = recs_by_user.get(user_id, [])
        if not recs or not all(has_valid_title(i) for i in recs):
            continue

        history_items = user_history_items(user_id, target_domain)
        records.append({
            'user_id': user_id,
            'domain': target_domain,
            'history_titles': [title_for(i) for i in history_items],
            'recommended_titles': [title_for(i) for i in recs],
            'held_out_item': ground_truth.get(user_id, []),
            'held_out_title': [title_for(i) for i in ground_truth.get(user_id, [])],
        })
        if len(records) >= n_target:
            break

    print(f"{target_domain}: {len(records)}/{n_target} clean records built")
    return records

books_eval_records = build_clean_eval_records('books')
movies_eval_records = build_clean_eval_records('movies')

books: 200 candidates passed history filter (from 500 drawn)
  Processed 0/200 users for books...
books: 100/100 clean records built
movies: 187 candidates passed history filter (from 500 drawn)
  Processed 0/187 users for movies...
movies: 100/100 clean records built


Retrieving the key and install the SDK

In [22]:
!pip install -q anthropic

from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [24]:
key = os.environ.get("ANTHROPIC_API_KEY")
print(f"key present: {key is not None}, length: {len(key) if key else 0}, starts with: {key[:7] if key else None}")

key present: True, length: 73, starts with: sk-or-v


Running on some 5 users to test

In [46]:
import anthropic
import json

client = anthropic.Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"],
    base_url="https://openrouter.ai/api",
)

JUDGE_MODEL = "anthropic/claude-haiku-4.5"

JUDGE_SYSTEM_PROMPT = """You are evaluating a book/movie recommender system.
Given a user's interaction history and a list of 10 recommended titles, rate how
well each recommendation fits the user's apparent tastes.

Score each recommendation 1-5:
1 = no plausible connection to the user's history
3 = plausible (same broad category, but doesn't necessarily align with the user's taste)
5 = strong fit with the user's apparent taste

Respond with ONLY valid JSON, no other text, in this exact format:
{"scores": [{"title": "...", "score": <int 1-5>, "reason": "<one short phrase>"}, ...]}
"""

def build_judge_prompt(record):
    history = "\n".join(f"- {t}" for t in record['history_titles']) or "(no history available)"
    recs = "\n".join(f"{i+1}. {t}" for i, t in enumerate(record['recommended_titles']))
    return f"""User's recent history ({record['domain']}):
{history}

Recommended titles to evaluate:
{recs}"""

def judge_user(record):
    response = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=1000,
        system=JUDGE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": build_judge_prompt(record)}],
    )
    raw_text = response.content[0].text.strip()
    raw_text = raw_text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError:
        print(f"  [parse failed for user {record['user_id']}] raw output:\n{raw_text}")
        return None
    return parsed

# --- test batch: 5 users only ---
test_batch = books_eval_records[:5]
judged_results = []

for record in test_batch:
    result = judge_user(record)
    judged_results.append({'user_id': record['user_id'], 'judgment': result})
    if result:
        avg = sum(s['score'] for s in result['scores']) / len(result['scores'])
        print(f"user {record['user_id']}: avg score {avg:.2f}")
    print("---")

judged_results[0]

user AE5RFTHXB5MFLYK6RO6LF5MD2RXA: avg score 3.10
---
user AHOXYXHTQCVKADMCFO2KNTW3OIHA: avg score 3.20
---
user AFT5MENJEQ57VRNLYJ7V6ULTTLGA: avg score 3.30
---
user AHZLNR3SRGAHEEZCRWABYFGQE7JA: avg score 1.90
---
user AH43AZNYO4WS3Y4CWLPTE3R2R27A: avg score 2.90
---


{'user_id': 'AE5RFTHXB5MFLYK6RO6LF5MD2RXA',
 'judgment': {'scores': [{'title': 'Frozen (Little Golden Book)',
    'score': 5,
    'reason': 'Disney animated film format matches Moana preference'},
   {'title': 'First 100 Words: A Padded Board Book',
    'score': 3,
    'reason': 'Board book for young children, but lacks narrative story'},
   {'title': 'Potty (Leslie Patricelli board books)',
    'score': 2,
    'reason': 'Potty training book without character-driven storytelling'},
   {'title': 'P is for Potty! (Sesame Street) (Lift-the-Flap)',
    'score': 2,
    'reason': 'Educational potty book, minimal character narrative appeal'},
   {'title': 'Dear Zoo: A Lift-the-Flap Book',
    'score': 3,
    'reason': 'Interactive format appeals to age group but lacks established characters'},
   {'title': 'Puppy Birthday to You! (Paw Patrol) (Little Golden Book)',
    'score': 4,
    'reason': "Popular children's character series in Little Golden Book format"},
   {'title': 'The New Baby',
 

Running on 100 users per domain

In [47]:
import time

def run_judge_batch(eval_records, domain_label, save_path, sleep_between=0.0):
    results = []
    for i, record in enumerate(eval_records):
        result = judge_user(record)
        results.append({'user_id': record['user_id'], 'domain': domain_label, 'judgment': result})

        if (i + 1) % 25 == 0:
            print(f"  {domain_label}: {i+1}/{len(eval_records)} judged")
            with open(save_path, 'w') as f:
                json.dump(results, f)

        if sleep_between:
            time.sleep(sleep_between)

    with open(save_path, 'w') as f:
        json.dump(results, f)
    return results

books_judged = run_judge_batch(books_eval_records, 'books', f'{DATA_DIR}/books_judged.json')
movies_judged = run_judge_batch(movies_eval_records, 'movies', f'{DATA_DIR}/movies_judged.json')

def summarize(judged_results):
    all_scores = []
    parse_failures = 0
    for r in judged_results:
        if r['judgment'] is None:
            parse_failures += 1
            continue
        all_scores.extend(s['score'] for s in r['judgment']['scores'])

    print(f"n_users={len(judged_results)}, parse_failures={parse_failures}, "
          f"n_scored_recs={len(all_scores)}, "
          f"mean_score={sum(all_scores)/len(all_scores):.3f}, "
          f"pct_score_ge_4={sum(1 for s in all_scores if s >= 4)/len(all_scores)*100:.1f}%")

print("Books:")
summarize(books_judged)
print("Movies:")
summarize(movies_judged)

  books: 25/100 judged
  books: 50/100 judged
  books: 75/100 judged
  books: 100/100 judged
  movies: 25/100 judged
  movies: 50/100 judged
  movies: 75/100 judged
  movies: 100/100 judged
Books:
n_users=100, parse_failures=0, n_scored_recs=1000, mean_score=2.823, pct_score_ge_4=37.8%
Movies:
n_users=100, parse_failures=0, n_scored_recs=1000, mean_score=3.534, pct_score_ge_4=58.1%


Optional things if you want to look at the specific users

In [49]:
# to inspect a specific user
def inspect_user(user_id, eval_records, judged_results):
    record = next(r for r in eval_records if r['user_id'] == user_id)
    judged = next(j for j in judged_results if j['user_id'] == user_id)

    print(f"User: {user_id}  (domain: {record['domain']})\n")
    print("History:")
    for t in record['history_titles']:
        print(f"  - {t}")

    print("\nRecommendations + judge scores:")
    for s in judged['judgment']['scores']:
        print(f"  [{s['score']}] {s['title']}")
        print(f"        -> {s['reason']}")

    print(f"\nHeld-out (ground truth) item: {record['held_out_title']}")

inspect_user('AE5RFTHXB5MFLYK6RO6LF5MD2RXA', books_eval_records, judged_results)

User: AE5RFTHXB5MFLYK6RO6LF5MD2RXA  (domain: books)

History:
  - Llama Llama Misses Mama
  - When I Grow Up (Little Critter) (Look-Look)
  - Little Critter: Just A Storybook Collection
  - Being Thankful (Mercer Mayer's Little Critter)
  - Moana Little Golden Book (Disney Moana)
  - Lyle, Lyle, Crocodile Storybook Treasury (Lyle the Crocodile)
  - Poppy's Party (DreamWorks Trolls) (Step into Reading)

Recommendations + judge scores:
  [5] Frozen (Little Golden Book)
        -> Disney animated film format matches Moana preference
  [3] First 100 Words: A Padded Board Book
        -> Board book for young children, but lacks narrative story
  [2] Potty (Leslie Patricelli board books)
        -> Potty training book without character-driven storytelling
  [2] P is for Potty! (Sesame Street) (Lift-the-Flap)
        -> Educational potty book, minimal character narrative appeal
  [3] Dear Zoo: A Lift-the-Flap Book
        -> Interactive format appeals to age group but lacks established charac

In [48]:
# to look specifically at the users with low ratings
# def avg_score(judged_entry):
#     if judged_entry['judgment'] is None:
#         return None
#     scores = [s['score'] for s in judged_entry['judgment']['scores']]
#     return sum(scores) / len(scores)


# movies_with_avg = [
#     {'user_id': j['user_id'], 'avg': avg_score(j)}
#     for j in movies_judged
# ]
# movies_with_avg = [m for m in movies_with_avg if m['avg'] is not None]
# movies_with_avg.sort(key=lambda m: m['avg'])


# print("Lowest-scoring movie users:")
# for m in movies_with_avg[:5]:
#     print(f"  {m['user_id']}: {m['avg']:.2f}")


# for m in movies_with_avg[:4]:
#     inspect_user(m['user_id'], movies_eval_records, movies_judged)